# Exploratory: referral frontiers for partner choice (does any cost ratio favour the least-correlated partner?)

For each deployed model (anchor), two partners are chosen on estimation data: the **strongest** (lowest severity-weighted miss count) and the **least error-correlated** (binary error correlation). For each, missed PDR is traced against total referral rate across a range of operating points, for:

- the anchor alone;
- the union rule (either model refers), each model at the same fraction f;
- the matched-volume ensemble (max percentile rank, referring fraction f).

Curves are interpolated onto a common referral-rate grid, so every comparison is at equal referral volume. If the least-correlated partner's curve never lies below the strongest partner's in missed PDR, it loses at every cost ratio of a missed PDR to a false positive. Also reported: break-even cost ratios (extra referrals per extra PDR eye caught).

- **EyePACS:** picks from the estimation pool; curves on the confirmation pool.
- **DDR:** picks from half A, curves on half B; and the swap.

Exploratory; outcome data previously opened. CPU, High-RAM, about 15 to 20 minutes.

In [1]:
# Cell 1: setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, itertools, datetime
import numpy as np, pandas as pd
from scipy.stats import rankdata
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, PREDS, RESULTS, FIGS = (f'{BASE}/{d}' for d in ('01_splits','03_predictions','04_results','05_figures'))
os.makedirs(FIGS, exist_ok=True)
ROSTER = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
          'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
          'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS = ['', '_seed1', '_seed2']; W = {4: 500, 3: 200, 2: 15}
N_BOOT, RNG = 2000, 20260927
def pick(df, c):
    for x in c:
        if x in df.columns: return x
    raise KeyError(f'{c} not in {list(df.columns)}')
def load_preds(ids, tpl):
    P = {}
    for a in ROSTER:
        for s in SEEDS:
            df = pd.read_csv(tpl.format(m=f'{a}{s}'), dtype=str)
            idc = pick(df, ['image_id']); pc = pick(df, ['predicted_probability','probability','prob'])
            v = df.drop_duplicates(subset=idc).set_index(idc)[pc].astype(float).reindex(ids)
            assert v.notna().all(); P[f'{a}{s}'] = v.to_numpy(np.float64)
    return P
def refer(score, frac, mask=None):
    n = len(score); idx = np.arange(n) if mask is None else np.where(mask)[0]
    k = int(round(frac * len(idx))); m = np.zeros(n, bool); m[idx[np.argsort(-score[idx], kind='stable')[:k]]] = True; return m
def pct(score, mask=None):
    n = len(score); out = np.zeros(n); idx = np.arange(n) if mask is None else np.where(mask)[0]
    out[idx] = rankdata(score[idx]) / len(idx); return out
def combos(a, b): return [(f'{a}{sa}', f'{b}{sb}') for sa in SEEDS for sb in SEEDS]
def picks_from(P, grade, frac, mask=None):
    n = len(grade); keep = np.ones(n, bool) if mask is None else mask; referable = grade >= 2
    Wm = np.zeros(n); [Wm.__setitem__(grade == g, w) for g, w in W.items()]
    ref = {k: refer(v, frac, mask) for k, v in P.items()}
    marg = {a: float(np.mean([Wm[referable & ~ref[f'{a}{s}'] & keep].sum() for s in SEEDS])) for a in ROSTER}
    rho = {}
    for a, b in itertools.combinations(ROSTER, 2):
        r = [np.corrcoef((referable != ref[ka])[keep].astype(float), (referable != ref[kb])[keep].astype(float))[0, 1] for ka, kb in combos(a, b)]
        rho[(a, b)] = rho[(b, a)] = float(np.nanmean(r))
    out = {}
    for a in ROSTER:
        c = [x for x in ROSTER if x != a]
        out[a] = {'strongest': min(c, key=lambda x: marg[x]), 'least_correlated': min(c, key=lambda x: rho[(a, x)])}
    return out
print('ready')

Mounted at /content/drive
ready


In [2]:
# Cell 2: frontier machinery
def curves_for_anchor(P, grade, a, partners, fracs, mask):
    '''Returns per-rule arrays of (referral_rate, pdr_missed, fp), averaged over seeds, per fraction.'''
    n_eval = mask.sum(); referable = grade >= 2; pdr = grade == 4
    ref = {}; rk = {}
    def R(k, f):
        if (k, f) not in ref: ref[(k, f)] = refer(P[k], f, mask)
        return ref[(k, f)]
    def RK(k):
        if k not in rk: rk[k] = pct(P[k], mask)
        return rk[k]
    out = {'single': []}
    for f in fracs:
        pts = [((R(f'{a}{s}', f)).sum() / n_eval, (pdr & ~R(f'{a}{s}', f) & mask).sum(), (~referable & R(f'{a}{s}', f) & mask).sum()) for s in SEEDS]
        out['single'].append(np.mean(pts, axis=0))
    for lab, b in partners.items():
        out[f'union_{lab}'] = []; out[f'matched_{lab}'] = []
        for f in fracs:
            u, m = [], []
            for ka, kb in combos(a, b):
                x = R(ka, f) | R(kb, f); u.append((x.sum() / n_eval, (pdr & ~x & mask).sum(), (~referable & x & mask).sum()))
                e = refer(np.maximum(RK(ka), RK(kb)), f, mask); m.append((e.sum() / n_eval, (pdr & ~e & mask).sum(), (~referable & e & mask).sum()))
            out[f'union_{lab}'].append(np.mean(u, axis=0)); out[f'matched_{lab}'].append(np.mean(m, axis=0))
    return {k: np.array(v) for k, v in out.items()}

def interp_on(grid, arr, col):
    x, y = arr[:, 0], arr[:, col]; o = np.argsort(x)
    v = np.interp(grid, x[o], y[o], left=np.nan, right=np.nan); return v

def site_frontier(name, P, grade, mask, picks, fracs):
    per = {a: curves_for_anchor(P, grade, a, picks[a], fracs, mask) for a in ROSTER}
    rules = list(per[ROSTER[0]].keys())
    lo = max(max(per[a][r][:, 0].min() for r in rules) for a in ROSTER); hi = min(min(per[a][r][:, 0].max() for r in rules) for a in ROSTER)
    grid = np.round(np.linspace(lo, hi, 25), 4)
    M = {r: np.array([interp_on(grid, per[a][r], 1) for a in ROSTER]) for r in rules}   # anchors x grid, PDR missed
    FP = {r: np.array([interp_on(grid, per[a][r], 2) for a in ROSTER]) for r in rules}
    rng = np.random.default_rng(RNG); idx = [rng.integers(0, len(ROSTER), len(ROSTER)) for _ in range(N_BOOT)]
    def diff(r1, r2):
        d = M[r1] - M[r2]; mean = np.nanmean(d, axis=0)
        bs = np.array([np.nanmean(d[i], axis=0) for i in idx])
        return {'mean': np.round(mean, 3).tolist(), 'lo': np.round(np.nanpercentile(bs, 2.5, axis=0), 3).tolist(),
                'hi': np.round(np.nanpercentile(bs, 97.5, axis=0), 3).tolist(),
                'grid_points_first_better': int((mean < 0).sum()), 'grid_points_first_better_ci_excludes_zero': int((np.nanpercentile(bs, 97.5, axis=0) < 0).sum())}
    res = {'grid_referral_rate': grid.tolist(),
           'mean_pdr_missed': {r: np.round(np.nanmean(M[r], axis=0), 3).tolist() for r in rules},
           'union_least_correlated_minus_union_strongest': diff('union_least_correlated', 'union_strongest'),
           'matched_least_correlated_minus_matched_strongest': diff('matched_least_correlated', 'matched_strongest'),
           'union_strongest_minus_single': diff('union_strongest', 'single'),
           'matched_strongest_minus_single': diff('matched_strongest', 'single'),
           'union_least_correlated_minus_matched_strongest': diff('union_least_correlated', 'matched_strongest')}
    # Break-even ratios at each site's base fraction (the fraction closest to the locked operating point)
    base_i = int(np.argmin(np.abs(np.array(fracs) - BASE_F[name])))
    def at_base(r): return np.mean([per[a][r][base_i] for a in ROSTER], axis=0)
    s, uS, uL = at_base('single'), at_base('union_strongest'), at_base('union_least_correlated')
    res['break_even_at_base_fraction'] = {
        'base_fraction_each_model': fracs[base_i],
        'union_strongest_vs_single': {'extra_pdr_caught': round(float(s[1] - uS[1]), 3), 'extra_fp': round(float(uS[2] - s[2]), 1),
                                      'fp_per_pdr': round(float((uS[2] - s[2]) / max(s[1] - uS[1], 1e-9)), 1)},
        'union_least_corr_vs_union_strongest': {'extra_pdr_caught': round(float(uS[1] - uL[1]), 3), 'extra_fp': round(float(uL[2] - uS[2]), 1),
                                                'fp_per_pdr': round(float((uL[2] - uS[2]) / (uS[1] - uL[1])), 1) if uS[1] - uL[1] > 0 else 'dominated'}}
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    for r, st in (('single', 'k-'), ('union_strongest', 'b-'), ('union_least_correlated', 'r-'), ('matched_strongest', 'b--'), ('matched_least_correlated', 'r--')):
        ax.plot(grid, np.nanmean(M[r], axis=0), st, label=r.replace('_', ' '))
    ax.set_xlabel('referral rate'); ax.set_ylabel('PDR eyes missed (mean over deployed models)'); ax.set_title(name); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(f'{FIGS}/frontier_{name}_v2.png', dpi=200); plt.close(fig)
    return res
print('ready')

ready


In [3]:
# Cell 3: EyePACS (picks on estimation pool, curves on confirmation pool)
ec = pd.read_csv(f'{SPLITS}/eyepacs_split_kaggletest_estconf_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
pool = ec[pick(ec, ['pool','split','set','assignment','estconf'])].astype(str).str.lower()
gE = ec[pick(ec, ['level','grade'])].to_numpy().astype(int)
PE = load_preds(ec.image_id.to_numpy(), PREDS + '/eyepacs_predictions_kaggletest_{m}_v2.csv')
EST = pool.str.startswith('est').to_numpy(); CONF = pool.str.startswith('conf').to_numpy()
BASE_F = {'EyePACS': 0.25}
pk_E = picks_from(PE, gE, 0.25, EST)
RES = {'EyePACS': site_frontier('EyePACS', PE, gE, CONF, pk_E, list(np.round(np.arange(0.05, 0.625, 0.025), 3)))}
print(json.dumps(RES['EyePACS']['break_even_at_base_fraction'], indent=1))
for k in ('union_least_correlated_minus_union_strongest', 'matched_least_correlated_minus_matched_strongest'):
    v = RES['EyePACS'][k]; print(k, '| least-corr better at', v['grid_points_first_better'], 'of 25 rates; CI excludes zero at', v['grid_points_first_better_ci_excludes_zero'])

{
 "base_fraction_each_model": 0.25,
 "union_strongest_vs_single": {
  "extra_pdr_caught": 13.848,
  "extra_fp": 927.4,
  "fp_per_pdr": 67.0
 },
 "union_least_corr_vs_union_strongest": {
  "extra_pdr_caught": 0.727,
  "extra_fp": 231.2,
  "fp_per_pdr": 317.9
 }
}
union_least_correlated_minus_union_strongest | least-corr better at 8 of 25 rates; CI excludes zero at 2
matched_least_correlated_minus_matched_strongest | least-corr better at 8 of 25 rates; CI excludes zero at 0


In [4]:
# Cell 4: DDR (picks on one half, curves on the other; both directions)
dd = pd.read_csv(f'{SPLITS}/ddr_split_intraset_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
gD = dd.grade.to_numpy().astype(int); A = (dd.half == 'A').to_numpy(); B = ~A
PD = load_preds(dd.image_id.to_numpy(), PREDS + '/ddr_predictions_{m}_v2.csv')
F = 1.31 * float((gD[A] >= 2).mean()); BASE_F.update({'DDR_A_to_B': F, 'DDR_B_to_A': F})
fr_D = list(np.round(np.arange(0.20, 0.875, 0.025), 3))
RES['DDR_A_to_B'] = site_frontier('DDR_A_to_B', PD, gD, B, picks_from(PD, gD, F, A), fr_D)
RES['DDR_B_to_A'] = site_frontier('DDR_B_to_A', PD, gD, A, picks_from(PD, gD, F, B), fr_D)
for s in ('DDR_A_to_B', 'DDR_B_to_A'):
    print(s, json.dumps(RES[s]['break_even_at_base_fraction']))
    for k in ('union_least_correlated_minus_union_strongest', 'matched_least_correlated_minus_matched_strongest'):
        v = RES[s][k]; print('  ', k, '| least-corr better at', v['grid_points_first_better'], 'of 25; CI excludes zero at', v['grid_points_first_better_ci_excludes_zero'])

/tmp/ipykernel_1776/2635952482.py:45: RuntimeWarning: Mean of empty slice
  'mean_pdr_missed': {r: np.round(np.nanmean(M[r], axis=0), 3).tolist() for r in rules},
/tmp/ipykernel_1776/2635952482.py:39: RuntimeWarning: Mean of empty slice
  d = M[r1] - M[r2]; mean = np.nanmean(d, axis=0)
/tmp/ipykernel_1776/2635952482.py:40: RuntimeWarning: Mean of empty slice
  bs = np.array([np.nanmean(d[i], axis=0) for i in idx])
/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1650: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,
/tmp/ipykernel_1776/2635952482.py:63: RuntimeWarning: Mean of empty slice
  ax.plot(grid, np.nanmean(M[r], axis=0), st, label=r.replace('_', ' '))


DDR_A_to_B {"base_fraction_each_model": 0.55, "union_strongest_vs_single": {"extra_pdr_caught": 15.111, "extra_fp": 342.8, "fp_per_pdr": 22.7}, "union_least_corr_vs_union_strongest": {"extra_pdr_caught": 0.263, "extra_fp": 37.9, "fp_per_pdr": 144.2}}
   union_least_correlated_minus_union_strongest | least-corr better at 7 of 25; CI excludes zero at 2
   matched_least_correlated_minus_matched_strongest | least-corr better at 7 of 25; CI excludes zero at 2
DDR_B_to_A {"base_fraction_each_model": 0.55, "union_strongest_vs_single": {"extra_pdr_caught": 13.283, "extra_fp": 405.8, "fp_per_pdr": 30.6}, "union_least_corr_vs_union_strongest": {"extra_pdr_caught": 1.162, "extra_fp": 91.1, "fp_per_pdr": 78.4}}
   union_least_correlated_minus_union_strongest | least-corr better at 9 of 25; CI excludes zero at 3
   matched_least_correlated_minus_matched_strongest | least-corr better at 8 of 25; CI excludes zero at 3


In [5]:
# Cell 5: write
out = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
       'status': 'exploratory; all outcome data previously opened under locks v4 and v5.1',
       'reading': 'All differences are in PDR eyes missed at equal total referral rate (negative = first rule misses fewer). If least-correlated never misses fewer than strongest at any referral rate, it is dominated at every cost ratio.',
       'results': RES}
with open(f'{RESULTS}/frontier_partner_choice_v2.json', 'w') as f: json.dump(out, f, indent=2, default=float)
print('Wrote 04_results/frontier_partner_choice_v2.json and three figures in 05_figures')

Wrote 04_results/frontier_partner_choice_v2.json and three figures in 05_figures
